In [ ]:
#test_landsat_histogram creates and plots histogram of rgb values. 
#Illustrates difference between scaled and unscaled images for two locations: default San Francisco, and Johns Hopkins Glacier
#CONCLUSION: SR arrives as unscaled large integers, TOA arrives scaled from 0-1.5

#TODO: apply this to test_landsat_geemap


#CONCLUSION: saves histograms in C:\Users\andyb\Documents\U\GEE-Courses\data
#For San Francisco test:
#SR scaled:     Red [0.000,0.748] Green [0.004,0.693] Blue [0.000,0.634] GOOD
#SR unscaled:   Red [271.000,28356.000] Green [1496.000,30342.000] Blue [5450.000,32469.000] BAD
#TOA scaled:    Red [0.000,0.000] Green [0.000,0.000] Blue [0.000,0.000] FAIL
#TOA unscaled:  Red [0.100,0.571] Green [0.074,0.612] Blue [0.042,0.626] GOOD
#For Johns Hopkins Glacier test:
#SR scaled:     Red [0.000,1.503] Green [0.004,1.513] Blue [0.058,1.511]
#SR unscaled:   Red [2769.000,60506.000] Green [14.000,62227.000] Blue [710.000,62293.000]
#TOA scaled:    Red [0.000,0.000] Green [0.000,0.000] Blue [0.000,0.000]
#TOA unscaled:  Red [0.218,1.229] Green [0.169,1.295] Blue [0.097,1.268]

#NOTE: gamma adjusts brightness of shadows/darker parts.

import ee
import geemap
import matplotlib.pyplot as plt
import numpy as np
import os

# Initialize Earth Engine (authenticate if needed)
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

folder_fig=r'C:\Users\andyb\Documents\U\GEE-Courses\data'

In [ ]:
# Step 1: Define a region of interest (ROI) - example: San Francisco area
roi = ee.Geometry.Rectangle([-122.5, 37.7, -122.3, 37.9])  # [lon_min, lat_min, lon_max, lat_max]

# Step 2: Load and filter Landsat 8 SR collection
collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')  # Surface Reflectance, Tier 1
              .filterBounds(roi)
              .filterDate('2014-03-01', '2014-05-31')  # Summer 2020 for low clouds
              .filterMetadata('CLOUD_COVER', 'less_than', 10)  # Low cloud cover
              .sort('CLOUD_COVER')  # Sort by cloud cover
              .first())  # Select the first (clearest) image. Use .median() for a composite instead.
collectionT = (ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')  # Top of Atmosphere, Tier 1
              .filterBounds(roi)
              .filterDate('2014-03-01', '2014-05-31')  # Summer 2020 for low clouds
              .filterMetadata('CLOUD_COVER', 'less_than', 10)  # Low cloud cover
              .sort('CLOUD_COVER')  # Sort by cloud cover
              .first())  # Select the first (clearest) image. Use .median() for a composite instead.

print(f'Selected image ID: {collection.get("system:index").getInfo()}, TOA: {collectionT.get("system:index").getInfo()}')
#print(f'Collection size: {collection.size().getInfo()}') #error since collection is only 1 image (.first arguement)
#known good image::: image = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_044034_20140318')

In [ ]:
#Glacier ROI (overwrites cell above)
# Step 1: Define a region of interest (ROI) - example: San Francisco area
roi = ee.Geometry.Rectangle([-137.148,58.824,-137.091,58.846])  # [lon_min, lat_min, lon_max, lat_max]

# Step 2: Load and filter Landsat 8 SR collection
collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')  # Surface Reflectance, Tier 1
              .filterBounds(roi)
              .filterDate('2014-03-01', '2014-05-31')  # Summer 2020 for low clouds
              .filterMetadata('CLOUD_COVER', 'less_than', 10)  # Low cloud cover
              .sort('CLOUD_COVER')  # Sort by cloud cover
              .first())  # Select the first (clearest) image. Use .median() for a composite instead.
collectionT = (ee.ImageCollection('LANDSAT/LC08/C02/T1_TOA')  # Top of Atmosphere, Tier 1
              .filterBounds(roi)
              .filterDate('2014-03-01', '2014-05-31')  # Summer 2020 for low clouds
              .filterMetadata('CLOUD_COVER', 'less_than', 10)  # Low cloud cover
              .sort('CLOUD_COVER')  # Sort by cloud cover
              .first())  # Select the first (clearest) image. Use .median() for a composite instead.

print(f'Selected image ID: {collection.get("system:index").getInfo()}, TOA: {collectionT.get("system:index").getInfo()}')
#print(f'Collection size: {collection.size().getInfo()}') #error since collection is only 1 image (.first arguement)
#known good image::: image = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_044034_20140318')

In [ ]:
# Step 3: Select RGB bands and apply scaling (multiply by 0.0000275 and add -0.2 for reflectance)
rgb_bands = ['SR_B4', 'SR_B3', 'SR_B2']  # Red, Green, Blue
rgb_image = collection.select(rgb_bands).multiply(0.0000275).add(-0.2)
rgb_image = rgb_image.updateMask(rgb_image.gte(0))  # Mask negative values (artifacts)
rgb_image

In [ ]:
rgb_bandsT = ['B4', 'B3', 'B2']  # Red, Green, Blue
rgb_imageT = collectionT.select(rgb_bandsT).multiply(0.0000275).add(-0.2)
rgb_imageT = rgb_imageT.updateMask(rgb_imageT.gte(0))  # Mask negative values (artifacts)
rgb_imageT

In [ ]:
# Step 4: Create an interactive map for visualization (optional)
#m = geemap.Map(center=[37.8,-122.4], zoom=11)
m = geemap.Map(center=[58.8,-137.11], zoom=11)
#vis_params = {'bands': rgb_bands, 'min': 0, 'max': 0.3, 'gamma': 1.2}  # Quick RGB viz (unscaled)
#vis_paramsT = {'bands': rgb_bandsT, 'min': 0, 'max': 0.3, 'gamma': 1.2}  # Quick RGB viz (unscaled)
vis_params = {'bands': rgb_bands, 'min': 0, 'max': 1.4, 'gamma': 1.2}  # Quick RGB viz (unscaled)
vis_paramsT = {'bands': rgb_bandsT, 'min': 0, 'max': 1.4, 'gamma': 1.2}  # Quick RGB viz (unscaled)
#NOTE: Grok labeled line above as "unscaled" but in fact this is operating on the scaled version.
vis_params2 = {'bands': rgb_bands, 'min': 0.1, 'max': 0.3, 'gamma': 1.2}  # Quick RGB viz (unscaled)
#m.addLayer(collection.clip(roi), vis_params, 'unscaled SR RGB') #display is all white
m.addLayer(collection.clip(roi), {}, 'unscaled SR RGB') #quite dark and grey if SF, not bad JHI
m.addLayer(rgb_image,vis_params,'scaled SR RGB') #works with vis_params GOOD
m.addLayer(collectionT.clip(roi), {}, 'unscaled TOA RGB') #dark and grey in SF, not bad JHI - highlights washed out a bit
m.addLayer(collectionT.clip(roi), vis_paramsT, 'unscaled vis TOA RGB') #works, brighter than SR version in SF, JHI similar to SR
m.addLayer(rgb_imageT,vis_paramsT,'scaled TOA RGB') #FAIL - zeros
m.addLayer(rgb_imageT,{},'scaled2 TOA RGB') #FAIL - zeros
#m.addLayerControl()
m  # Displays the map in Jupyter/Colab

In [ ]:
#SR scaled
# Step 5: Sample pixels to NumPy array for local histogram plotting
# Use a region slightly larger than ROI for better sampling; scale=30m for Landsat resolution
array = geemap.ee_to_numpy(rgb_image, region=roi.buffer(1000), scale=30) 
#, default_value=0) #Exception: Invalid JSON payload received. Unknown name "default_value": Cannot find field.

# Flatten the array to 1D (exclude nodata/masked values)
red = array[:,:,0].flatten()
green = array[:,:,1].flatten()
blue = array[:,:,2].flatten()

# Remove nodata values (0 or NaN)
valid_mask = (red > 0) & (green > 0) & (blue > 0) & (~np.isnan(red)) & (~np.isnan(green)) & (~np.isnan(blue))
red = red[valid_mask]
green = green[valid_mask]
blue = blue[valid_mask]

print(f'Sampled {len(red)} valid pixels. Red [{red.min():.3f},{red.max():.3f}] Green [{green.min():.3f},{green.max():.3f}] Blue [{blue.min():.3f},{blue.max():.3f}]')

# Step 6: Plot histograms
plt.figure(figsize=(10, 6))
plt.hist(red, bins=100, alpha=0.7, label='Red (SR_B4)', color='red', density=True)
plt.hist(green, bins=100, alpha=0.7, label='Green (SR_B3)', color='green', density=True)
plt.hist(blue, bins=100, alpha=0.7, label='Blue (SR_B2)', color='blue', density=True)
plt.xlabel('Reflectance Value')
plt.ylabel('Density')
plt.title('Histogram of scaled SR RGB Values from Landsat 8 Image')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(folder_fig,'Histogram Scaled SR.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#TOA scaled
# Step 5: Sample pixels to NumPy array for local histogram plotting
# Use a region slightly larger than ROI for better sampling; scale=30m for Landsat resolution
array = geemap.ee_to_numpy(rgb_imageT, region=roi.buffer(1000), scale=30) 
#, default_value=0) #Exception: Invalid JSON payload received. Unknown name "default_value": Cannot find field.

# Flatten the array to 1D (exclude nodata/masked values)
red = array[:,:,0].flatten()
green = array[:,:,1].flatten()
blue = array[:,:,2].flatten()

# Remove nodata values (0 or NaN)
#valid_mask = (red > 0) & (green > 0) & (blue > 0) & (~np.isnan(red)) & (~np.isnan(green)) & (~np.isnan(blue))
#red = red[valid_mask]
#green = green[valid_mask]
#blue = blue[valid_mask]

print(f'Sampled {len(red)} valid pixels. Red [{red.min():.3f},{red.max():.3f}] Green [{green.min():.3f},{green.max():.3f}] Blue [{blue.min():.3f},{blue.max():.3f}]')

# Step 6: Plot histograms
plt.figure(figsize=(10, 6))
plt.hist(red, bins=100, alpha=0.7, label='Red (B4)', color='red', density=True)
plt.hist(green, bins=100, alpha=0.7, label='Green (B3)', color='green', density=True)
plt.hist(blue, bins=100, alpha=0.7, label='Blue (B2)', color='blue', density=True)
plt.xlabel('Reflectance Value')
plt.ylabel('Density')
plt.title('Histogram of scaled TOA RGB Values from Landsat 8 Image')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(folder_fig,'Histogram Scaled TOA.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#SR unscaled
#now for unscaled version
# Step 5: Sample pixels to NumPy array for local histogram plotting
# Use a region slightly larger than ROI for better sampling; scale=30m for Landsat resolution
array = geemap.ee_to_numpy(collection, region=roi.buffer(1000), scale=30) 
#, default_value=0) #Exception: Invalid JSON payload received. Unknown name "default_value": Cannot find field.

# Flatten the array to 1D (exclude nodata/masked values)
red = array[:,:,0].flatten()
green = array[:,:,1].flatten()
blue = array[:,:,2].flatten()

# Remove nodata values (0 or NaN)
valid_mask = (red > 0) & (green > 0) & (blue > 0) & (~np.isnan(red)) & (~np.isnan(green)) & (~np.isnan(blue))
red = red[valid_mask]
green = green[valid_mask]
blue = blue[valid_mask]

print(f'Sampled {len(red)} valid pixels. Red [{red.min():.3f},{red.max():.3f}] Green [{green.min():.3f},{green.max():.3f}] Blue [{blue.min():.3f},{blue.max():.3f}]')

# Step 6: Plot histograms
plt.figure(figsize=(10, 6))
plt.hist(red, bins=100, alpha=0.7, label='Red (SR_B4)', color='red', density=True)
plt.hist(green, bins=100, alpha=0.7, label='Green (SR_B3)', color='green', density=True)
plt.hist(blue, bins=100, alpha=0.7, label='Blue (SR_B2)', color='blue', density=True)
plt.xlabel('Reflectance Value')
plt.ylabel('Density')
plt.title('Histogram of unscaled SR RGB Values from Landsat 8 Image')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(folder_fig,'Histogram Unscaled SR.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#TOA unscaled
#now for unscaled version
# Step 5: Sample pixels to NumPy array for local histogram plotting
# Use a region slightly larger than ROI for better sampling; scale=30m for Landsat resolution
array = geemap.ee_to_numpy(collectionT, region=roi.buffer(1000), scale=30) 
#, default_value=0) #Exception: Invalid JSON payload received. Unknown name "default_value": Cannot find field.

# Flatten the array to 1D (exclude nodata/masked values)
red = array[:,:,0].flatten()
green = array[:,:,1].flatten()
blue = array[:,:,2].flatten()

# Remove nodata values (0 or NaN)
valid_mask = (red > 0) & (green > 0) & (blue > 0) & (~np.isnan(red)) & (~np.isnan(green)) & (~np.isnan(blue))
red = red[valid_mask]
green = green[valid_mask]
blue = blue[valid_mask]

print(f'Sampled {len(red)} valid pixels. Red [{red.min():.3f},{red.max():.3f}] Green [{green.min():.3f},{green.max():.3f}] Blue [{blue.min():.3f},{blue.max():.3f}]')

# Step 6: Plot histograms
plt.figure(figsize=(10, 6))
plt.hist(red, bins=100, alpha=0.7, label='Red (B4)', color='red', density=True)
plt.hist(green, bins=100, alpha=0.7, label='Green (B3)', color='green', density=True)
plt.hist(blue, bins=100, alpha=0.7, label='Blue (B2)', color='blue', density=True)
plt.xlabel('Reflectance Value')
plt.ylabel('Density')
plt.title('Histogram of unscaled TOA RGB Values from Landsat 8 Image')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(folder_fig,'Histogram Unscaled TOA.png'), dpi=300, bbox_inches='tight')
plt.show()